# Examination Hall Seating Allocation Simulator
**Course:** Solving Problems Using Modelling and Simulation (SPMS) - NEC00091  
**Students:** Krishna Shrivastava (BTEC25O1031), Aditya Mishra (BTEC25O1003)

## 1. Imports and setup

In [ ]:
import csv
import math
import os
import random
import time

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch

%matplotlib inline

BRANCHES = [
    ("ECE", "Electronics & Communication", "Signals & Systems", 0.28),
    ("EE", "Electrical Engineering", "Circuit Theory", 0.22),
    ("CSE", "Computer Science", "Data Structures", 0.24),
    ("ME", "Mechanical Engineering", "Thermodynamics", 0.16),
    ("CE", "Civil Engineering", "Surveying", 0.10),
]
HALLS = [("Hall A", 6, 8), ("Hall B", 6, 8), ("Hall C", 6, 8),
         ("Hall D", 6, 8), ("Hall E", 6, 8)]
COLORS = ["#2E86AB", "#F6AE2D", "#3BB273", "#E15554", "#7768AE"]
GREY = "#E6E6E6"

## 2. Students and hall layout

In [ ]:
def make_students(n):
    counts = [int(round(n * b[3])) for b in BRANCHES]
    counts[0] += n - sum(counts)
    students = []
    for k, c in enumerate(counts):
        for i in range(1, c + 1):
            students.append((BRANCHES[k][0] + "-" + str(i).zfill(3), k))
    return students


def total_seats():
    return sum(r * c for _, r, c in HALLS)


def neighbours(rows, cols):
    nb = [[] for _ in range(rows * cols)]
    for r in range(rows):
        for c in range(cols):
            i = r * cols + c
            if r > 0:
                nb[i].append(i - cols)
            if r < rows - 1:
                nb[i].append(i + cols)
            if c > 0:
                nb[i].append(i - 1)
            if c < cols - 1:
                nb[i].append(i + 1)
    return nb


def edges(rows, cols):
    u, v = [], []
    for r in range(rows):
        for c in range(cols):
            i = r * cols + c
            if c < cols - 1:
                u.append(i)
                v.append(i + 1)
            if r < rows - 1:
                u.append(i)
                v.append(i + cols)
    return np.array(u), np.array(v)

## 3. Cost function (conflicts)

In [ ]:
def hall_conflicts(grid, rows, cols):
    u, v = edges(rows, cols)
    a = grid[u]
    b = grid[v]
    return int(np.sum((a == b) & (a >= 0)))


def bad_seats(grid, rows, cols):
    nb = neighbours(rows, cols)
    bad = np.zeros(rows * cols, dtype=bool)
    for i, v in enumerate(grid):
        if v >= 0 and any(grid[j] == v for j in nb[i]):
            bad[i] = True
    return bad


def papers_of(layout, students):
    return [np.array([students[s][1] if s >= 0 else -1 for s in hall])
            for hall in layout]


def count_conflicts(layout, students):
    total = 0
    for p, (_, r, c) in zip(papers_of(layout, students), HALLS):
        total += hall_conflicts(p, r, c)
    return total


def count_affected(layout, students):
    total = 0
    for p, (_, r, c) in zip(papers_of(layout, students), HALLS):
        total += int(bad_seats(p, r, c).sum())
    return total

## 4. Seating strategies: sequential, random, optimised

In [ ]:
def split_into_halls(flat):
    layout = []
    pos = 0
    for _, r, c in HALLS:
        layout.append(np.array(flat[pos:pos + r * c]))
        pos += r * c
    return layout


def sequential_layout(students):
    flat = list(range(len(students))) + [-1] * (total_seats() - len(students))
    return split_into_halls(flat)


def random_layout(students, rng):
    flat = list(range(len(students))) + [-1] * (total_seats() - len(students))
    rng.shuffle(flat)
    return split_into_halls(flat)


def starting_layout(students, rng):
    order = sorted(range(len(students)), key=lambda i: (students[i][1], rng.random()))
    groups = [[] for _ in HALLS]
    for k, s in enumerate(order):
        groups[k % len(HALLS)].append(s)
    layout = []
    for members, (_, r, c) in zip(groups, HALLS):
        flat = members + [-1] * (r * c - len(members))
        rng.shuffle(flat)
        layout.append(np.array(flat))
    return layout


def anneal(grid, rows, cols, rng, iters, t_start=2.0, t_end=0.02):
    s = list(grid)
    nb = neighbours(rows, cols)
    n = rows * cols

    def seat_cost(i):
        v = s[i]
        if v < 0:
            return 0
        return sum(1 for j in nb[i] if s[j] == v)

    current = hall_conflicts(np.array(s), rows, cols)
    history = [current]
    for k in range(iters):
        temp = t_start * (t_end / t_start) ** (k / iters)
        a, b = rng.sample(range(n), 2)
        if s[a] != s[b]:
            before = seat_cost(a) + seat_cost(b)
            s[a], s[b] = s[b], s[a]
            delta = seat_cost(a) + seat_cost(b) - before
            if delta <= 0 or rng.random() < math.exp(-delta / temp):
                current += delta
            else:
                s[a], s[b] = s[b], s[a]
        if (k + 1) % 50 == 0:
            history.append(current)
    return np.array(s), history


def optimised_layout(students, rng, iters):
    start = starting_layout(students, rng)
    papers = papers_of(start, students)
    result = []
    hist = None
    for hall, pap, (_, r, c) in zip(start, papers, HALLS):
        new_pap, h = anneal(pap, r, c, rng, iters)
        pools = {}
        for s in hall:
            if s >= 0:
                pools.setdefault(students[s][1], []).append(s)
        seats = np.array([pools[p].pop() if p >= 0 else -1 for p in new_pap])
        result.append(seats)
        hist = np.array(h) if hist is None else hist + np.array(h)
    return result, hist

## 5. Saving and drawing

In [ ]:
def save_csv(layout, students, path):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["Hall", "Row", "Seat", "Roll No", "Branch", "Paper"])
        for hall, (name, r, c) in zip(layout, HALLS):
            for i, s in enumerate(hall):
                if s >= 0:
                    k = students[s][1]
                    w.writerow([name, i // c + 1, i % c + 1, students[s][0],
                                BRANCHES[k][0], BRANCHES[k][2]])


def draw_hall(ax, grid, rows, cols, title):
    bad = bad_seats(grid, rows, cols)
    for i, v in enumerate(grid):
        r, c = divmod(i, cols)
        color = COLORS[v] if v >= 0 else GREY
        ax.add_patch(Rectangle((c, rows - 1 - r), 0.92, 0.92, color=color))
        if bad[i]:
            ax.add_patch(Rectangle((c, rows - 1 - r), 0.92, 0.92, fill=False,
                                   edgecolor="red", linewidth=2.2))
        if v >= 0:
            ax.text(c + 0.46, rows - 1 - r + 0.46, BRANCHES[v][0], ha="center",
                    va="center", fontsize=6.5, color="white", fontweight="bold")
    ax.set_xlim(-0.1, cols)
    ax.set_ylim(-0.1, rows)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, fontsize=10)


def legend_items():
    items = [Patch(color=COLORS[k], label=b[0] + " - " + b[2]) for k, b in enumerate(BRANCHES)]
    items.append(Patch(color=GREY, label="Empty seat"))
    items.append(Patch(fill=False, edgecolor="red", linewidth=2, label="Same-paper neighbour"))
    return items


def make_graphs(out, students, lay_seq, lay_rand, lay_opt, hist, mc, res):
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.6))
    _, r, c = HALLS[0]
    names = ["Sequential", "Random", "Optimised (Annealing)"]
    for ax, lay, name in zip(axs, [lay_seq, lay_rand, lay_opt], names):
        p = papers_of(lay, students)[0]
        draw_hall(ax, p, r, c, f"Hall A - {name}\nconflicts: {hall_conflicts(p, r, c)}")
    fig.legend(handles=legend_items(), loc="lower center", ncol=7, fontsize=8, frameon=False)
    fig.suptitle("Seating of Hall A under three strategies", fontsize=13, fontweight="bold")
    fig.tight_layout(rect=[0, 0.07, 1, 0.94])
    fig.savefig(os.path.join(out, "fig1_strategy_comparison_hallA.png"), dpi=150)
    plt.show()

    fig, axs = plt.subplots(2, 3, figsize=(14, 9.5))
    papers = papers_of(lay_opt, students)
    for k, (ax, (name, r, c)) in enumerate(zip(axs.flat, HALLS)):
        used = int((lay_opt[k] >= 0).sum())
        draw_hall(ax, papers[k], r, c, f"{name} ({used}/{r * c} seats used)")
    for ax in list(axs.flat)[len(HALLS):]:
        ax.axis("off")
    axs.flat[-1].legend(handles=legend_items(), loc="center", fontsize=9, frameon=False)
    fig.suptitle("Optimised seating plan - all halls", fontsize=14, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.95], h_pad=3.0)
    fig.savefig(os.path.join(out, "fig2_optimised_all_halls.png"), dpi=150)
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(np.arange(len(hist)) * 50, hist, color="#2E86AB", linewidth=2)
    ax.set_xlabel("Annealing iterations (per hall)")
    ax.set_ylabel("Total conflicts (all halls)")
    ax.set_title("Simulated Annealing convergence", fontweight="bold")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "fig3_convergence.png"), dpi=150)
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(mc, bins=25, color="#F6AE2D", edgecolor="white")
    ax.axvline(res["Sequential"]["conflicts"], color="#E15554", linewidth=2,
               label=f"Sequential = {res['Sequential']['conflicts']}")
    ax.axvline(np.mean(mc), color="black", linestyle="--", linewidth=2,
               label=f"Random mean = {np.mean(mc):.1f}")
    ax.axvline(res["Optimised"]["conflicts"], color="#3BB273", linewidth=2,
               label=f"Optimised = {res['Optimised']['conflicts']}")
    ax.set_xlabel("Conflicts in one random seating")
    ax.set_ylabel("Number of trials")
    ax.set_title(f"Monte Carlo simulation ({len(mc)} random seatings)", fontweight="bold")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(out, "fig4_monte_carlo.png"), dpi=150)
    plt.show()

    fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
    labels = ["Sequential", "Random (mean)", "Optimised"]
    colors = ["#E15554", "#F6AE2D", "#3BB273"]
    conf = [res["Sequential"]["conflicts"], float(np.mean(mc)), res["Optimised"]["conflicts"]]
    bars = axs[0].bar(labels, conf, color=colors)
    axs[0].bar_label(bars, fmt="%.1f")
    axs[0].set_ylabel("Conflicting neighbour pairs")
    axs[0].set_title("Conflicts by strategy", fontweight="bold")
    aff = [res["Sequential"]["affected"], res["Random"]["affected"], res["Optimised"]["affected"]]
    bars = axs[1].bar(labels, aff, color=colors)
    axs[1].bar_label(bars, fmt="%.0f")
    axs[1].set_ylabel("Students next to same paper")
    axs[1].set_title("Students affected", fontweight="bold")
    for a in axs:
        a.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "fig5_summary.png"), dpi=150)
    plt.show()

## 6. Run the simulation
Change the numbers below if you want a different run (maximum students = 240).

In [ ]:
NUM_STUDENTS = 200
TRIALS = 500
ITERS = 6000
SEED = 42
OUT = "output"
FIND_ROLL = "ECE-001"

if NUM_STUDENTS < 1 or NUM_STUDENTS > total_seats():
    raise ValueError(f"NUM_STUDENTS must be between 1 and {total_seats()}")

os.makedirs(OUT, exist_ok=True)
rng = random.Random(SEED)
students = make_students(NUM_STUDENTS)

print("=" * 64)
print(" EXAMINATION HALL SEATING ALLOCATION SIMULATOR")
print("=" * 64)
print(f" Students : {len(students)}   Halls : {len(HALLS)}   Seats : {total_seats()}")
for k, b in enumerate(BRANCHES):
    n = sum(1 for s in students if s[1] == k)
    print(f"   {b[0]:<4}{b[2]:<20}{n:>4} students")

lay_seq = sequential_layout(students)
mc = []
mc_aff = []
for _ in range(TRIALS):
    lay = random_layout(students, rng)
    mc.append(count_conflicts(lay, students))
    mc_aff.append(count_affected(lay, students))
lay_rand = random_layout(students, rng)

t0 = time.time()
lay_opt, hist = optimised_layout(students, rng, ITERS)
t_opt = time.time() - t0

res = {
    "Sequential": {"conflicts": count_conflicts(lay_seq, students),
                   "affected": count_affected(lay_seq, students)},
    "Random": {"conflicts": float(np.mean(mc)), "affected": float(np.mean(mc_aff))},
    "Optimised": {"conflicts": count_conflicts(lay_opt, students),
                  "affected": count_affected(lay_opt, students)},
}

seated = [s for hall in lay_opt for s in hall if s >= 0]
assert sorted(seated) == list(range(len(students))), "allocation error"

print("\n RESULTS")
print(f" {'Strategy':<22}{'Conflicts':>10}{'Students affected':>20}")
print(" " + "-" * 52)
print(f" {'Sequential':<22}{res['Sequential']['conflicts']:>10}{res['Sequential']['affected']:>20}")
label = "Random (mean of " + str(TRIALS) + ")"
print(f" {label:<22}{res['Random']['conflicts']:>10.1f}{res['Random']['affected']:>20.1f}")
print(f" {'Optimised (Annealing)':<22}{res['Optimised']['conflicts']:>10}{res['Optimised']['affected']:>20}")
print(f"\n Best random trial : {min(mc)} conflicts | Worst : {max(mc)}")
print(f" Annealing time    : {t_opt:.2f} s")
reduction = 100 * (1 - res["Optimised"]["conflicts"] / max(res["Random"]["conflicts"], 1e-9))
print(f" Reduction vs random seating: {reduction:.1f}%")

save_csv(lay_opt, students, os.path.join(OUT, "seating_plan.csv"))
print(f"\n Saved seating_plan.csv in '{OUT}/'")

## 7. Graphs

In [ ]:
make_graphs(OUT, students, lay_seq, lay_rand, lay_opt, hist, mc, res)

## 8. Find a student's seat

In [ ]:
def find_student(roll):
    key = roll.strip().upper()
    for hall, (name, r, c) in zip(lay_opt, HALLS):
        for i, s in enumerate(hall):
            if s >= 0 and students[s][0] == key:
                print(f" {key} -> {name}, Row {i // c + 1}, Seat {i % c + 1}")
                return
    print(f" Roll number '{key}' not found.")


find_student(FIND_ROLL)